In [ ]:
#Importing libraries
#Running Sentiment Analysis using Textblob and Vader

In [2]:
import pandas as pd
from textblob import TextBlob
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# ---------------------------------------------------------
# SETUP: Ensure VADER lexicon is downloaded (Run this once)
# ---------------------------------------------------------
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

# Load the file, preserving "N/A"
filename = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Combined_Castello.csv"
df = pd.read_csv(filename, keep_default_na=False)

# Ensure empty cells are "N/A"
df = df.replace(r'^\s*$', 'N/A', regex=True)

# ---------------------------------------------------------
# FIX: Only use columns that really exist
# ---------------------------------------------------------
possible_cols = ['title', 'description', 'one_comments', 'tags']

existing_cols = [c for c in possible_cols if c in df.columns]

print("Using text columns:", existing_cols)

def get_clean_text(row):
    parts = []
    for col in existing_cols:
        val = str(row[col])
        if val.strip() != 'N/A':
            parts.append(val)
    return " ".join(parts)

# ---------------------------------------------------------
# 1. TextBlob Analysis
# ---------------------------------------------------------
def get_textblob_score(row):
    text = get_clean_text(row)
    if not text.strip():
        return 0.0
    return TextBlob(text).sentiment.polarity

# ---------------------------------------------------------
# 2. VADER Analysis
# ---------------------------------------------------------
sid = SentimentIntensityAnalyzer()

def get_vader_score(row):
    text = get_clean_text(row)
    if not text.strip():
        return 0.0
    return sid.polarity_scores(text)['compound']

# Apply both functions
df['TextBlob_Score'] = df.apply(get_textblob_score, axis=1)
df['Vader_Score'] = df.apply(get_vader_score, axis=1)

# Save to new CSV
output = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Castello_Both_Scores.csv"
df.to_csv(output, index=False)

print("Saved to:", output)
print(df[['TextBlob_Score', 'Vader_Score']].head())


Using text columns: ['title', 'tags']
Saved to: C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Castello_Both_Scores.csv
   TextBlob_Score  Vader_Score
0            0.00       0.0772
1            0.08       0.0000
2            0.00       0.0000
3            0.00       0.0000
4            0.00       0.0000


In [ ]:
#Running sentiment analysis on pictures using Clip

In [3]:
import sys
import io

# ---------------------------------------------------------
# DEPENDENCY CHECK & SAFE IMPORT
# ---------------------------------------------------------
# We attempt to import libraries inside try-except blocks.
# If they fail, we add them to a list and exit gracefully with instructions.

missing_libraries = []

try:
    import pandas as pd
except ImportError:
    missing_libraries.append("pandas")

try:
    import requests
except ImportError:
    missing_libraries.append("requests")

try:
    from PIL import Image
    from io import BytesIO
except ImportError:
    missing_libraries.append("pillow")

try:
    import torch
except ImportError:
    missing_libraries.append("torch")

try:
    from transformers import CLIPProcessor, CLIPModel
except ImportError:
    missing_libraries.append("transformers")

if missing_libraries:
    print("---------------------------------------------------------")
    print("⚠️  MISSING LIBRARIES DETECTED")
    print("---------------------------------------------------------")
    print(f"The following libraries are missing: {', '.join(missing_libraries)}")
    print("\nPlease run this command in your terminal/command prompt:")
    print(f"\n    pip install {' '.join(missing_libraries)}\n")
    print("---------------------------------------------------------")
    sys.exit(0)

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
INPUT_FILE = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Castello_Both_Scores.csv"
OUTPUT_FILE = r"C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Castello_Sentiment_CLIP.csv"
MODEL_ID = "openai/clip-vit-base-patch32"

# ---------------------------------------------------------
# VENICE SENTIMENT PROMPTS
# CLIP compares the image to these text descriptions.
# ---------------------------------------------------------
POSITIVE_PROMPTS = [
    "a beautiful sunny day in Venice",
    "a romantic gondola ride",
    "peaceful canal water",
    "historic italian architecture",
    "happy people enjoying food",
    "colorful buildings in Venice",
    "artistic and aesthetic photo"
]

NEGATIVE_PROMPTS = [
    "a crowded and stressful tourist trap",
    "garbage and trash on the ground",
    "gloomy grey weather",
    "dirty water",
    "construction work and scaffolding",
    "blurry low quality photo",
    "flooded street acqua alta"
]

ALL_PROMPTS = POSITIVE_PROMPTS + NEGATIVE_PROMPTS

# ---------------------------------------------------------
# INITIALIZE MODEL
# ---------------------------------------------------------
print(f"Loading CLIP model ({MODEL_ID})...")
try:
    model = CLIPModel.from_pretrained(MODEL_ID)
    processor = CLIPProcessor.from_pretrained(MODEL_ID)
except Exception as e:
    print(f"Error loading model: {e}")
    sys.exit(1)

def analyze_image_clip(url):
    """
    Downloads image and compares it against positive/negative text prompts.
    Returns:
        - Most likely description
        - Sentiment Score (-1.0 to 1.0)
    """
    if str(url).strip().lower() == 'n/a' or str(url).strip() == '':
        return "N/A", 0.0

    try:
        # 1. Download Image
        response = requests.get(url, stream=True, timeout=5)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))
        
        # 2. Process inputs
        inputs = processor(
            text=ALL_PROMPTS, 
            images=image, 
            return_tensors="pt", 
            padding=True
        )

        # 3. Run Inference
        with torch.no_grad():
            outputs = model(**inputs)
            
        # 4. Calculate Probabilities
        # logits_per_image: [1, len(ALL_PROMPTS)]
        probs = outputs.logits_per_image.softmax(dim=1)
        probs_list = probs[0].tolist() # Convert to standard python list

        # 5. Calculate Sentiment Score
        # Sum of probabilities for positive prompts minus sum for negative prompts
        # Result will be naturally between -1 and 1
        pos_score = sum(probs_list[:len(POSITIVE_PROMPTS)])
        neg_score = sum(probs_list[len(POSITIVE_PROMPTS):])
        
        final_sentiment = pos_score - neg_score
        
        # Get the single highest matching text description
        max_prob_index = probs_list.index(max(probs_list))
        best_description = ALL_PROMPTS[max_prob_index]

        return best_description, round(final_sentiment, 4)

    except Exception as e:
        # print(f"Error: {e}") 
        return "Error", 0.0

# ---------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------
print(f"Reading {INPUT_FILE}...")
try:
    df = pd.read_csv(INPUT_FILE, keep_default_na=False)
except FileNotFoundError:
    print(f"❌ Error: {INPUT_FILE} not found.")
    print("Please ensure the CSV file is in the same directory.")
    sys.exit(1)

df = df.replace(r'^\s*$', 'N/A', regex=True)

clip_desc_list = []
clip_score_list = []

total_rows = len(df)
print(f"Processing {total_rows} images using CLIP (this may be slower than YOLO)...")

for index, row in df.iterrows():
    url = row.get('url_s', 'N/A')
    
    if index % 5 == 0:
        print(f"Processing row {index}/{total_rows}...")
        
    desc, score = analyze_image_clip(url)
    
    clip_desc_list.append(desc)
    clip_score_list.append(score)

# Add new columns
df['CLIP_Best_Desc'] = clip_desc_list
df['CLIP_Visual_Score'] = clip_score_list

# Save
df.to_csv(OUTPUT_FILE, index=False)
print("---------------------------------------------------------")
print(f"Analysis Complete. Saved to {OUTPUT_FILE}")
try:
    print(df[['title', 'CLIP_Best_Desc', 'CLIP_Visual_Score']].head())
except KeyError:
    pass

c:\Users\TUF\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading CLIP model (openai/clip-vit-base-patch32)...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Reading C:\Users\TUF\Desktop\Architecture\UCL\Design\General_Flickr_Scrapping\Castello\Castello_Both_Scores.csv...
Processing 5692 images using CLIP (this may be slower than YOLO)...
Processing row 0/5692...
Processing row 5/5692...
Processing row 10/5692...
Processing row 15/5692...
Processing row 20/5692...
Processing row 25/5692...
Processing row 30/5692...
Processing row 35/5692...
Processing row 40/5692...
Processing row 45/5692...
Processing row 50/5692...
Processing row 55/5692...
Processing row 60/5692...
Processing row 65/5692...
Processing row 70/5692...
Processing row 75/5692...
Processing row 80/5692...
Processing row 85/5692...
Processing row 90/5692...
Processing row 95/5692...
Processing row 100/5692...
Processing row 105/5692...
Processing row 110/5692...
Processing row 115/5692...
Processing row 120/5692...
Processing row 125/5692...
Processing row 130/5692...
Processing row 135/5692...
Processing row 140/5692...
Processing row 145/5692...
Processing row 150/5692...
Pr

KeyboardInterrupt: 